# Production plan with stock at the plants

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/plant_stock.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the first version of the model, M0, the demand of a month is assembled in that month, and with forecast 2 the model has no solution, although the plants could meet that demand with bikes assembled in advance. This notebook uses the second version, M1, in which each plant holds stock from one month to the next.

Contents of the notebook:

1. Plant data and forecast 2.
2. Linear model with stock at the plants.
3. Optimal plan under forecast 2, with and without the required final stock.
4. Four tests, each with one datum changed: the holding cost, the demand of one month, the planning horizon and the required final stock.

The cells run in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders, drop-down lists and tick boxes): after changing a value, run the cell again. Outside Colab the form is a line such as `last_month = "Mar"`: edit the value and run the cell.

The table gives the symbols of M0 and M1. The column «From» gives the version where each symbol first appears.

| Symbol | Meaning | Unit | From |
|---|---|---|---|
| $F$, $f$ | Plants | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | M0 |
| $\tau$ | Last month of the horizon | — | M1 |
| $c_f$ | Unit assembly cost at plant $f$ | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | bikes/month | M0 |
| $d_t$ | Orders received in month $t$ | bikes/month | M0 |
| $c^h_f$ | Holding cost at plant $f$ | €/(bike·month) | M1 |
| $h_{f0}$ | Initial stock at plant $f$, at the end of December | bikes | M1 |
| $h^{\text{end}}_f$ | Required final stock: the stock that plant $f$ must hold at the end of the horizon | bikes | M1 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | bikes/month | M0 |
| $h_{ft}$ | Bikes in stock at plant $f$ at the end of month $t$ | bikes | M1 |
| $x_{ft}$ | Bikes shipped from plant $f$ in month $t$ | bikes/month | M1 |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/plant_stock.md) explains all the data and results. The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import plant_stock as ps

## 1. Plant data and forecast 2

The first table gives, for each plant, the unit assembly cost (€/bike) and the capacity (bikes/month, the same every month) of M0, and the three data that M1 adds: the holding cost of one bike for one month, the initial stock, at the end of December, and the required final stock, equal to the initial stock. The second table is forecast 2 of M0, with which M0 has no solution.

In [ ]:
# @title Plant data and forecast 2 { display-mode: "form" }
ps.show_data()

## 2. Linear model with stock at the plants

$$
\begin{aligned}
\min \quad & \sum_{f \in F} \sum_{t \in T} \left( c_f\, p_{ft} + c^h_f\, h_{ft} \right) && && \text{(assembly and holding cost)} \\
\text{s.t.} \quad & h_{f,t-1} + p_{ft} = x_{ft} + h_{ft} && \forall f \in F,\ t \in T && \text{(stock balance at each plant)} \\
& x_{ft} \le h_{f,t-1} + p_{ft} && \forall f \in F,\ t \in T && \text{(shipment limit at each plant)} \\
& \sum_{f \in F} x_{ft} \ge d_t && \forall t \in T && \text{(demand of each month)} \\
& p_{ft} \le K_{ft} && \forall f \in F,\ t \in T && \text{(capacity)} \\
& h_{f\tau} \ge h^{\text{end}}_f && \forall f \in F && \text{(required final stock; only in the runs that require it)} \\
& p_{ft},\ x_{ft},\ h_{ft} \ge 0 && \forall f \in F,\ t \in T
\end{aligned}
$$

with $h_{f0}$ given. In the stock balance, what a plant holds at the start of a month plus what it assembles equals what it ships plus what it holds at the end of the month. The shipment limit says that a plant cannot ship more than it held at the start of the month plus what it assembles that month. The plants together ship at least the demand of each month. In M0 what a plant assembles in a month is what it delivers that month; with stock the two quantities differ, and the model has one variable for each.

## 3. Optimal plan under forecast 2

The cell solves the model with forecast 2 and gives, for each plant and month, the bikes assembled, the bikes shipped and the bikes in stock at the end of the month, and the cost of the plan. Below, the **shadow price** of the demand of each month (€/bike): what one more bike of demand that month would cost. The figure on the left stacks the bikes assembled by each plant against the capacity of the three plants and the demand; the one on the right stacks the stock of each plant at the end of each month, from December.

The tick box adds or removes the required final stock. Run the cell with and without it. In which months do the two plans differ, and why do the plants end June with no stock without it? Which plant holds the seasonal stock built before April, and why that one?

In [ ]:
# @title Optimal plan under forecast 2 { display-mode: "form" }
with_required_final_stock = True  # @param {type:"boolean"}
ps.stock_plan(with_required_final_stock);

## 4. Holding cost and seasonal stock

The slider sets the holding cost at plant 1, in €/(bike·month). The holding costs of plants 2 and 3 move with it, 1 and 2 €/(bike·month) higher, so that plant 1 remains the cheapest place to hold stock, and in the optimal plan the seasonal stock is held there. Seasonal stock costs its holding cost; the other way to meet April and May is plant 3, whose assembly cost is 50 €/bike above that of plant 2. The cell gives the seasonal stock at the end of March and the bikes assembled at plant 3, and then the plan as in the previous cell.

Raise the holding cost from 5. At what holding cost does plant 3 start to assemble in April, and which bikes of seasonal stock does it replace? How far in advance do plants 1 and 2 assemble when holding is expensive?

In [ ]:
# @title Optimal plan for a holding cost { display-mode: "form" }
holding_cost_plant_1 = 5  # @param {type:"slider", min:1, max:40, step:1}
ps.holding_cost_test(holding_cost_plant_1);

## 5. Demand of one month and infeasibility

Choose a month and its demand; the other months keep forecast 2, and the plan keeps the required final stock. While the demand from January to each month does not exceed what the plants can assemble in those months plus their initial stock, the plants assemble in advance to meet it. When it exceeds it, the model has no solution, and the cell gives the month of the unmet demand and the number of bikes.

Raise the demand of April step by step. Up to what demand does the model have a solution, and in which months do the plants assemble the extra bikes? When there is no solution, does the company have no way to meet that demand, or does the model leave out something it could do?

In [ ]:
# @title Optimal plan for a demand of one month { display-mode: "form" }
month = "Apr"  # @param ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
demand = 650  # @param {type:"slider", min:0, max:1500, step:10}
ps.demand_test(month, demand);

## 6. Planning horizon

The cell makes a plan from January to the month you choose, with the required final stock at the end of that month. Then it makes a second plan for the months left until June, from the stock the first plan leaves. With June there is only one plan, that of section 3.

Choose March. Do the plants build seasonal stock for April in the plan from January to March? What happens to the plan from April to June? Try April and May: how long does the horizon of the first plan have to be for it to include the peak of demand?

In [ ]:
# @title Optimal plans for a planning horizon { display-mode: "form" }
last_month = "Mar"  # @param ["Mar", "Apr", "May", "Jun"]
ps.horizon_test(last_month);

## 7. Required final stock

The required final stock is not decided by this plan: it comes from outside, for example from the plan for the following months. Choose a plant and its required final stock; the other plants keep theirs. The cell gives the plan with forecast 2 and that requirement, or the reason why there is no solution.

Raise the required final stock of plant 1. In which months and at which plants is the extra stock assembled? How much does the cost of the plan grow for each extra bike required?

In [ ]:
# @title Optimal plan for a required final stock { display-mode: "form" }
plant = "Plant 1"  # @param ["Plant 1", "Plant 2", "Plant 3"]
required_final_stock = 20  # @param {type:"slider", min:0, max:300, step:10}
ps.final_stock_test(plant, required_final_stock);

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/plant_stock.md) explains all the numbers and has a script that prints them.